In [0]:
customer = spark.read.table("ecommerce_analytics.silver.customers")

In [0]:
from pyspark.sql.functions import *
cust = customer



# Adding necessary columns

cust = cust.withColumn("customer_type",
                       when (col("loyalty_segment") == 2, "Regular")
                       .when (col("loyalty_segment") >= 3, "Premium")
                       .otherwise("New"))

cust = cust.withColumn("created_date",to_date(from_unixtime(col("valid_from"))))


# Selecting final required columns
cust = cust.select('customer_id',
        'first_name',
        'last_name',
        concat_ws(' ', 'first_name', 'last_name').alias('full_name'),
        'loyalty_segment',
        'customer_type',
        'created_date',
        to_date(col('last_update_ts')).alias("last_updated")
    ).dropDuplicates(['customer_id'])

# saveing the dataframe as a table
cust.write.mode("overwrite").saveAsTable("ecommerce_analytics.gold.dim_customer")